# Polarized Trees — Benchmark, Demo, and Inference

This notebook has **two roles**:

- it is a simple end-to-end example of how to use `PolarizedTreesBenchmark`;
- it is also the reproducibility code used for the synthetic benchmark reported in the paper.

The workflow is deliberately kept in one place:

**fixed synthetic data → hyperparameter search → best configuration → recovery evaluation → unseen inference → F/C/P**

The benchmark module performs the actual model selection. The notebook does not manually rank configurations or reconstruct the selected model.

> **Important:** the synthetic corpora must be generated first with `datasetdemo.ipynb`. This notebook reuses those fixed datasets so every configuration is evaluated on exactly the same data.


## What is fixed and what can be changed?

This notebook uses the **default benchmark settings provided by `polartox.benchmark`**. These defaults give a ready-to-run starting point, but they are not required.

The main benchmark settings can be changed:

- **search space** — pass a dictionary defining the values to consider for each hyperparameter;
- **strategy** — `"full"` evaluates every configuration, while `"random"` samples configurations;
- **number of runs** — controls how many configurations are sampled with random search;
- **seed** — makes random search reproducible;
- **metrics** — choose which recovery metrics to calculate;
- **selection metric** — choose which metric determines the best configuration;
- **selection direction** — choose `"max"` or `"min"`;
- **pipeline settings** — dimensions, scale, and any fixed pipeline parameters.

For this notebook, the default search space, metrics, and selection metric are imported directly from the package. You can replace `DEFAULT_SEARCH_SPACE` with your own dictionary without changing the benchmark code.

Ground truth is required for recovery-based model selection. After the best configuration is selected, the resulting pipeline can be run **without ground truth**, which is the normal inference setting for real annotation data.


## 1. Imports and experiment settings

Only a small amount of configuration is needed here. The actual search, evaluation, ranking, and selection are delegated to `PolarizedTreesBenchmark`.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

from polartox.benchmark import (
    PolarizedTreesBenchmark,
    DEFAULT_SEARCH_SPACE,
    DEFAULT_METRICS,
    DEFAULT_SELECTION_METRIC,
)
from polartox.pipeline import PolarizedTreesPipeline


# Locate the repository root.
CWD = Path.cwd().resolve()

if (CWD / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate project root. "
        "Run this notebook from the notebooks directory "
        "or the project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from benchmark_config import (
    DIMS,
    SCALE,
    BENCHMARK_CORPORA,
    INFERENCE_CORPUS,
)


DATA_DIR = PROJECT_ROOT / "benchmark_data"
RESULTS_DIR = PROJECT_ROOT / "benchmark_results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Paper settings.
N_RUNS = 800
SEED = 0

print("Project root:", PROJECT_ROOT)
print("Benchmark corpora:", BENCHMARK_CORPORA)
print("Inference corpus:", INFERENCE_CORPUS)
print("Random configurations:", N_RUNS)
print("Seed:", SEED)


Project root: C:\Users\swkra\OneDrive\Υπολογιστής\ερευνα\HumanModelAlign\polarizedtrees\benchmarks
Benchmark corpora: ['A_default', 'B_weak_signal', 'C_deep']
Inference corpus: inference_unseen
Random configurations: 800
Seed: 0


## 2. Load the fixed synthetic datasets

The first notebook generates these datasets once and stores both the annotations and the per-text ground truth.

Ground truth tells us which SCD dimensions actually generated the polarization. That makes recovery metrics possible and allows the benchmark to perform model selection.

The inference corpus is kept separate and is not used for selecting the configuration.


In [2]:
def load_corpus(name):
    dataset = pd.read_csv(
        DATA_DIR / f"{name}_dataset.csv"
    )

    with open(
        DATA_DIR / f"{name}_ground_truth.json",
        encoding="utf-8",
    ) as f:
        ground_truth = {
            int(text_id): value
            for text_id, value in json.load(f).items()
        }

    return dataset, ground_truth


corpora = {
    name: load_corpus(name)
    for name in BENCHMARK_CORPORA
}

inference_dataset, inference_ground_truth = load_corpus(
    INFERENCE_CORPUS
)


for name, (dataset, ground_truth) in corpora.items():
    print(
        f"{name}: "
        f"{dataset.shape[0]:,} annotations | "
        f"{dataset['text_id'].nunique()} texts | "
        f"{len(ground_truth)} ground-truth entries"
    )

print(
    f"{INFERENCE_CORPUS}: "
    f"{inference_dataset.shape[0]:,} annotations | "
    f"{inference_dataset['text_id'].nunique()} texts"
)


A_default: 162,000 annotations | 100 texts | 100 ground-truth entries
B_weak_signal: 162,000 annotations | 100 texts | 100 ground-truth entries
C_deep: 162,000 annotations | 100 texts | 100 ground-truth entries
inference_unseen: 162,000 annotations | 100 texts


## 3. Prepare the model-selection data

`PolarizedTreesBenchmark` works on one annotation dataset. We therefore combine A/B/C into one development corpus.

The text IDs are shifted so that the three corpora remain completely distinct. The three corpora contain the same number of texts, so their combined mean gives each corpus equal weight.

This is only a data-preparation step; it does not change the annotations or their ground truth.


In [3]:
text_counts = {
    name: dataset["text_id"].nunique()
    for name, (dataset, _) in corpora.items()
}

if len(set(text_counts.values())) != 1:
    raise ValueError(
        "A/B/C must contain the same number of texts "
        "for equal-weight mean Jaccard."
    )


benchmark_parts = []
benchmark_ground_truth = {}
text_groups = {}

offset = 0

for corpus_name in BENCHMARK_CORPORA:
    dataset, ground_truth = corpora[corpus_name]

    part = dataset.copy()

    text_ids = sorted(
        part["text_id"].unique()
    )

    id_map = {
        old_id: offset + i
        for i, old_id in enumerate(text_ids)
    }

    part["text_id"] = part["text_id"].map(id_map)

    benchmark_parts.append(part)

    for old_id, config in ground_truth.items():
        benchmark_ground_truth[
            id_map[old_id]
        ] = config

    for new_id in id_map.values():
        text_groups[new_id] = corpus_name

    offset += len(text_ids)


benchmark_dataset = pd.concat(
    benchmark_parts,
    ignore_index=True,
)

print("Combined annotations:", benchmark_dataset.shape)
print("Combined texts:", benchmark_dataset["text_id"].nunique())
print("Ground-truth entries:", len(benchmark_ground_truth))


Combined annotations: (486000, 8)
Combined texts: 300
Ground-truth entries: 300


## 4. Define the paper's search space

For the paper, we consider the full valid search space of 3,240 configurations.

The space is written as explicit configurations rather than a simple Cartesian product because `beta` is meaningful only when `variant="beta"`.

For a smaller demonstration, this cell can be replaced by a small custom search space. The benchmark API supports both forms.


In [4]:
SEARCH_SPACE = DEFAULT_SEARCH_SPACE.copy()

print("Search space:")
for parameter, values in SEARCH_SPACE.items():
    print(f"  {parameter}: {values}")

n_configurations = 1
for values in SEARCH_SPACE.values():
    n_configurations *= len(values)

print("\nTotal possible configurations:", n_configurations)


Search space:
  theta_filter: [0.2, 0.3, 0.4]
  min_size_frac: [0.02, 0.03, 0.05]
  max_depth: [4, 6, 8]
  variant: ['max', 'var', 'beta']
  h: [0.05, 0.1, 0.15, 0.2]
  relative_h: [False, True]
  theta_stop: [0.05, 0.1, 0.15]

Total possible configurations: 1944


## 5. Create the base Polarized Trees pipeline

The user provides **one pipeline object** to the benchmark.

The benchmark uses its settings as the base and creates candidate pipelines internally while testing configurations. After model selection, the selected pipeline is available directly through `benchmark.get_best_pipeline()`.

The user therefore does not need to manually reconstruct the winning pipeline.


In [5]:
# theta_filter, h and max_depth are required by the pipeline; the
# benchmark overrides all of them with each searched configuration,
# so these are placeholders.
pipeline = PolarizedTreesPipeline(
    dims=DIMS,
    scale=SCALE,
    theta_filter=0.3,
    h=0.10,
    max_depth=6,
)

print(pipeline)


## 6. Run model selection

This is the main benchmark step.

The benchmark:

1. generates the configurations from the supplied search-space dictionary;
2. evaluates the requested configurations on the same annotations and ground truth;
3. computes the requested recovery metrics;
4. ranks the configurations using the selected metric;
5. stores the best configuration and the corresponding pipeline.

Here we use random search so that the same notebook can also demonstrate the `random` strategy. For a small search space, `strategy="full"` is useful because it evaluates every possible configuration.

The defaults are imported from the package, but all of these choices can be changed.


In [6]:
benchmark = PolarizedTreesBenchmark(
    pipeline=pipeline,
    annotations=benchmark_dataset,
    ground_truth=benchmark_ground_truth,

    # Corpus of every text: metrics and statistics are computed per
    # corpus and then averaged over corpora.
    text_groups=text_groups,

    # Paper search.
    search_space=SEARCH_SPACE,
    strategy="random",
    n_runs=N_RUNS,
    seed=SEED,

    # Recovery metrics available in the synthetic setting.
    metrics=DEFAULT_METRICS,

    # Paper selection criterion.
    selection_metric=DEFAULT_SELECTION_METRIC,

    # Keep the highest score.
    selection_direction="max",

    top_k=20,
    verbose=True,

    # Save progress every 50 configurations; re-running this cell
    # after a crash resumes from the last checkpoint.
    checkpoint_dir=RESULTS_DIR / "checkpoint",
    checkpoint_every=50,

    # Evaluate configurations in parallel worker processes. Each worker
    # holds its own copy of the data (~150 MB); lower this if memory is tight.
    n_jobs=6,
)

benchmark.run()


[1/800] jaccard=0.7619
[2/800] jaccard=0.4231
[3/800] jaccard=0.8382
[4/800] jaccard=0.7439
[5/800] jaccard=0.8050
[6/800] jaccard=0.8720
[7/800] jaccard=0.8293
[8/800] jaccard=0.5178
[9/800] jaccard=0.8272
[10/800] jaccard=0.8382
[11/800] jaccard=0.8384
[12/800] jaccard=0.8414
[13/800] jaccard=0.8354
[14/800] jaccard=0.8691
[15/800] jaccard=0.8075
[16/800] jaccard=0.8154
[17/800] jaccard=0.5224
[18/800] jaccard=0.6666
[19/800] jaccard=0.8166
[20/800] jaccard=0.8298
[21/800] jaccard=0.8200
[22/800] jaccard=0.5328
[23/800] jaccard=0.8575
[24/800] jaccard=0.8288
[25/800] jaccard=0.8115
[26/800] jaccard=0.7879
[27/800] jaccard=0.8128
[28/800] jaccard=0.6950
[29/800] jaccard=0.8258
[30/800] jaccard=0.7790
[31/800] jaccard=0.6368
[32/800] jaccard=0.8436
[33/800] jaccard=0.8209
[34/800] jaccard=0.8200
[35/800] jaccard=0.8457
[36/800] jaccard=0.8282
[37/800] jaccard=0.5178
[38/800] jaccard=0.8185
[39/800] jaccard=0.6297
[40/800] jaccard=0.4694
[41/800] jaccard=0.8333
[42/800] jaccard=0.4694
[

## 7. Inspect the selected configuration

The benchmark now contains the complete model-selection result.

Nothing is selected manually here. These values are returned by the benchmark after ranking all evaluated configurations.


In [7]:
print("Best configuration:")
print(benchmark.get_best_config())

print("\nBest mean Jaccard:")
print(benchmark.get_best_score())

print("\nTop configurations:")
display(benchmark.get_top_configs())

Best configuration:
{'theta_filter': 0.3, 'min_size_frac': 0.03, 'max_depth': 4, 'variant': 'beta', 'beta': 1.0, 'h': 0.15, 'relative_h': True, 'theta_stop': 0.1}

Best mean Jaccard:
0.8917621987066432

Top configurations:


,rank,configuration_id,theta_filter,min_size_frac,max_depth,variant,beta,h,relative_h,theta_stop,...,precision_min,precision_max,recall,recall_median,recall_std,recall_q1,recall_q3,recall_min,recall_max,exact_match
0,1,152,0.3,0.03,4,beta,1.0,0.15,True,0.1,...,0.300000,1.0,0.931752,1.0,0.183839,0.916667,1.0,0.194444,1.0,0.747310
1,2,252,0.2,0.02,8,beta,1.0,0.15,True,0.1,...,0.300000,1.0,0.932267,1.0,0.182724,0.916667,1.0,0.194444,1.0,0.742754
2,3,564,0.2,0.02,4,beta,1.0,0.15,True,0.1,...,0.300000,1.0,0.932267,1.0,0.182724,0.916667,1.0,0.194444,1.0,0.742754
3,4,172,0.4,0.02,8,beta,1.0,0.15,True,0.1,...,0.300000,1.0,0.928151,1.0,0.185379,0.916667,1.0,0.194444,1.0,0.716428
4,5,779,0.3,0.02,4,beta,2.0,0.10,True,0.1,...,0.266667,1.0,0.934215,1.0,0.161052,0.916667,1.0,0.194444,1.0,0.712337
5,6,522,0.3,0.03,4,beta,0.5,0.20,True,0.1,...,0.222222,1.0,0.920742,1.0,0.204097,0.916667,1.0,0.083333,1.0,0.724043
6,7,80,0.3,0.02,4,beta,0.5,0.20,True,0.1,...,0.222222,1.0,0.920742,1.0,0.204097,0.916667,1.0,0.083333,1.0,0.724043
7,8,453,0.3,0.03,6,beta,1.0,0.10,True,0.1,...,0.266667,1.0,0.939946,1.0,0.159064,0.916667,1.0,0.194444,1.0,0.711500
8,9,143,0.3,0.02,8,beta,1.0,0.10,True,0.1,...,0.266667,1.0,0.939946,1.0,0.159064,0.916667,1.0,0.194444,1.0,0.711500
9,10,501,0.3,0.03,8,beta,1.0,0.10,True,0.1,...,0.266667,1.0,0.939946,1.0,0.159064,0.916667,1.0,0.194444,1.0,0.711500


## 8. Inspect all benchmark results

The complete table contains one row per evaluated configuration.

This is useful for checking how sensitive performance is to the different hyperparameters and for reproducing the ranking reported in the paper.


In [8]:
results_df = benchmark.get_results()

print("Evaluated configurations:", len(results_df))

display(
    results_df.head(20)
)


Evaluated configurations: 800


,rank,configuration_id,theta_filter,min_size_frac,max_depth,variant,beta,h,relative_h,theta_stop,...,precision_min,precision_max,recall,recall_median,recall_std,recall_q1,recall_q3,recall_min,recall_max,exact_match
0,1,152,0.3,0.03,4,beta,1.0,0.15,True,0.1,...,0.300000,1.0,0.931752,1.0,0.183839,0.916667,1.0,0.194444,1.0,0.747310
1,2,252,0.2,0.02,8,beta,1.0,0.15,True,0.1,...,0.300000,1.0,0.932267,1.0,0.182724,0.916667,1.0,0.194444,1.0,0.742754
2,3,564,0.2,0.02,4,beta,1.0,0.15,True,0.1,...,0.300000,1.0,0.932267,1.0,0.182724,0.916667,1.0,0.194444,1.0,0.742754
3,4,172,0.4,0.02,8,beta,1.0,0.15,True,0.1,...,0.300000,1.0,0.928151,1.0,0.185379,0.916667,1.0,0.194444,1.0,0.716428
4,5,779,0.3,0.02,4,beta,2.0,0.10,True,0.1,...,0.266667,1.0,0.934215,1.0,0.161052,0.916667,1.0,0.194444,1.0,0.712337
5,6,522,0.3,0.03,4,beta,0.5,0.20,True,0.1,...,0.222222,1.0,0.920742,1.0,0.204097,0.916667,1.0,0.083333,1.0,0.724043
6,7,80,0.3,0.02,4,beta,0.5,0.20,True,0.1,...,0.222222,1.0,0.920742,1.0,0.204097,0.916667,1.0,0.083333,1.0,0.724043
7,8,453,0.3,0.03,6,beta,1.0,0.10,True,0.1,...,0.266667,1.0,0.939946,1.0,0.159064,0.916667,1.0,0.194444,1.0,0.711500
8,9,143,0.3,0.02,8,beta,1.0,0.10,True,0.1,...,0.266667,1.0,0.939946,1.0,0.159064,0.916667,1.0,0.194444,1.0,0.711500
9,10,501,0.3,0.03,8,beta,1.0,0.10,True,0.1,...,0.266667,1.0,0.939946,1.0,0.159064,0.916667,1.0,0.194444,1.0,0.711500


## 9. Get the selected pipeline

The benchmark stores the winning pipeline itself.

This is the pipeline that should be used for the subsequent evaluation and inference steps. There is no need to instantiate another `PolarizedTreesPipeline` manually.


In [10]:
best_pipeline = benchmark.get_best_pipeline()

print(best_pipeline)


## 10. Evaluate the selected configuration on A/B/C

Model selection was performed on the combined development data. We now report the selected configuration separately on the three benchmark distributions.

Because ground truth is available, we can report Jaccard, precision, recall, and exact match.


In [11]:
# Per-corpus recovery of the selected (rank 1) configuration:
# mean plus distribution statistics of the per-text values,
# and the average over corpora.
group_results = benchmark.get_group_results()

selected_recovery = group_results[group_results["rank"] == 1]
selected_recovery = selected_recovery[
    ["corpus", "n_texts_evaluated", *benchmark._value_columns()]
].reset_index(drop=True)

mean_row = benchmark.get_results().iloc[[0]][
    benchmark._value_columns()
].assign(corpus="Mean")

selected_recovery = pd.concat(
    [selected_recovery, mean_row],
    ignore_index=True,
)

display(selected_recovery)

,corpus,n_texts_evaluated,jaccard,jaccard_median,jaccard_std,jaccard_q1,jaccard_q3,jaccard_min,jaccard_max,precision,...,precision_min,precision_max,recall,recall_median,recall_std,recall_q1,recall_q3,recall_min,recall_max,exact_match
0,A_default,84.0,0.918651,1.0,0.171602,1.000000,1.0,0.333333,1.0,0.974206,...,0.5,1.0,0.944444,1.0,0.152694,1.000000,1.0,0.333333,1.0,0.797619
1,B_weak_signal,81.0,0.869136,1.0,0.248623,0.750000,1.0,0.000000,1.0,0.875309,...,0.0,1.0,0.956790,1.0,0.192801,1.000000,1.0,0.000000,1.0,0.716049
2,C_deep,92.0,0.887500,1.0,0.212040,0.750000,1.0,0.250000,1.0,0.993478,...,0.4,1.0,0.894022,1.0,0.206023,0.750000,1.0,0.250000,1.0,0.728261
3,Mean,NaN,0.891762,1.0,0.210755,0.833333,1.0,0.194444,1.0,0.947664,...,0.3,1.0,0.931752,1.0,0.183839,0.916667,1.0,0.194444,1.0,0.747310


## 11. Run inference without ground truth

This is the important transition from **validation** to **inference**.

The selected configuration is now fixed. We deliberately do not pass ground truth.

In this setting Polarized Trees reports the outputs that are available on real annotation data:

- **F** — dimension frequency across tree depths;
- **C** — subgroup pole consistency;
- **P** — subgroup polarization reduction;
- diagnostics describing the resulting trees and residual polarization.

The ground-truth recovery metrics are not part of this inference result.


In [12]:
inference_results = best_pipeline.run_full_evaluation(
    inference_dataset,
    ground_truth=None,
    verbose=True,
)

F = inference_results["F"]
C = inference_results["C"]
P = inference_results["P"]
diagnostics = inference_results["diagnostics"]


print("=== F: Dimension Frequency ===")
display(F)

print("=== C: Subgroup Pole Consistency ===")
display(C.head(10))

print("=== P: Subgroup PRG ===")
display(P.head(10))

print("=== Diagnostics ===")
display(
    pd.Series(
        diagnostics,
        name="value",
    )
)


=== diagnostics ===
  retention_rate: 0.86
  mean_leaves: 7.593023255813954
  mean_depth: 2.3169984686064318
  mean_residual_ndfu: 0.17932784484016107
  mean_top_split_prg: 0.4408721547050139
  indeterminate_rate: 0.006125574272588055
  dims_never_used: []
=== F: Dimension Frequency ===


depth,1,2,3
dim,,,
age,16,22,21
education,17,29,20
gender,15,20,19
orientation,25,25,29
politics,13,25,27


=== C: Subgroup Pole Consistency ===


,n_s,frac_toxic,frac_civil
subgroup,,,
"((education, low),)",13,0.461538,0.538462
"((education, high),)",12,0.500000,0.500000
"((orientation, lgbtq+),)",12,0.500000,0.500000
"((orientation, heterosexual),)",12,0.583333,0.416667
"((education, medium),)",11,0.727273,0.272727
"((politics, center),)",8,0.500000,0.500000
"((politics, left),)",8,0.625000,0.375000
"((politics, right),)",7,0.714286,0.285714
"((age, 25-50), (education, high))",6,0.500000,0.500000


=== P: Subgroup PRG ===


,n_s,mean_prg
subgroup,,
"((age, >50), (gender, male), (orientation, lgbtq+))",2,0.679320
"((age, >50), (education, high), (politics, left))",1,0.643634
"((orientation, lgbtq+),)",12,0.635337
"((orientation, heterosexual),)",12,0.632869
"((education, low), (gender, non-binary), (politics, center))",1,0.623129
"((education, low), (gender, non-binary), (politics, right))",1,0.623129
"((education, low), (gender, female), (politics, right))",1,0.608743
"((education, low), (gender, female), (politics, center))",1,0.608743
"((education, low), (orientation, lgbtq+), (politics, center))",1,0.606818


=== Diagnostics ===


retention_rate            0.86
mean_leaves           7.593023
mean_depth            2.316998
mean_residual_ndfu    0.179328
mean_top_split_prg    0.440872
indeterminate_rate    0.006126
dims_never_used             []
Name: value, dtype: object

## 12. Save the benchmark and inference results

The benchmark report records the search settings, selected configuration, score, and top configurations.

The additional files provide the paper-facing recovery and inference outputs.


In [13]:
# Complete benchmark outputs.
# One row per configuration (averaged over corpora).
benchmark.save_results(
    RESULTS_DIR / "benchmark_configuration_summary.csv"
)

# One row per configuration and corpus.
benchmark.save_group_results(
    RESULTS_DIR / "benchmark_results.csv"
)

# Raw per-text recovery values (for boxplots).
benchmark.save_text_results(
    RESULTS_DIR / "benchmark_text_results.csv"
)

# Full report: settings, search space, best config, top configurations.
benchmark.save_report(
    RESULTS_DIR / "benchmark_report.json"
)

# Top-ranked configurations as a table.
benchmark.get_top_configs().to_csv(
    RESULTS_DIR / "top_configurations.csv",
    index=False,
)


# Selected configuration and recovery on A/B/C.
with open(
    RESULTS_DIR / "selected_configuration.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        benchmark.get_best_config(),
        f,
        indent=2,
    )

selected_recovery.to_csv(
    RESULTS_DIR / "selected_configuration_recovery.csv",
    index=False,
)


# Ground-truth-free inference outputs.
F.to_csv(
    RESULTS_DIR / "fcp_F_dimension_frequency.csv"
)

C.to_csv(
    RESULTS_DIR / "fcp_C_pole_consistency.csv"
)

P.to_csv(
    RESULTS_DIR / "fcp_P_subgroup_prg.csv"
)

with open(
    RESULTS_DIR / "fcp_inference_diagnostics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        benchmark._python_value(diagnostics),
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# Optional: everything inference-related in one Excel workbook.
try:
    with pd.ExcelWriter(
        RESULTS_DIR / "fcp_inference_results.xlsx"
    ) as writer:
        F.to_excel(writer, sheet_name="F")
        C.to_excel(writer, sheet_name="C")
        P.to_excel(writer, sheet_name="P")
        pd.Series(diagnostics, name="value").to_frame().to_excel(
            writer, sheet_name="diagnostics"
        )
except ImportError:
    print("openpyxl not installed: skipped fcp_inference_results.xlsx")


print(
    "Saved results to:",
    RESULTS_DIR.resolve()
)


Saved results to: C:\Users\swkra\OneDrive\Υπολογιστής\ερευνα\HumanModelAlign\polarizedtrees\benchmarks\benchmark_results


## 13. Adapting this notebook to another experiment

The notebook uses the package defaults as a convenient starting point. The benchmark itself is deliberately configurable.

The main things to change are:

```text
search_space        → a dictionary of hyperparameter values to search
strategy            → "full" or "random"
n_runs              → number sampled for random search
seed                → reproducible random sampling
metrics             → recovery metrics to compute
selection_metric    → metric used to choose the winner
selection_direction → "max" or "min"
pipeline            → fixed dimensions, scale, and other settings
annotations         → your annotation dataset
ground_truth        → required for recovery-based model selection
```

For example, a custom search can be as simple as:

```python
SEARCH_SPACE = {
    "relative_h": [True],
    "h": [0.05, 0.10, 0.15],
    "max_depth": [4, 6],
}
```

You then pass that dictionary directly to `PolarizedTreesBenchmark`.

If you already have an annotation dataset, you can use it directly as long as it follows the expected annotation format. Synthetic data is only needed when you want known ground truth for objective recovery evaluation.

If you do not have ground truth, the benchmark cannot use recovery metrics for model selection. In that case, use `PolarizedTreesPipeline` directly in inference mode and interpret F/C/P and diagnostics.

## Next steps

1. Run `datasetdemo.ipynb` to generate the fixed synthetic datasets.
2. Run this notebook to benchmark configurations and select the best pipeline.
3. Inspect the saved benchmark and inference outputs.
4. For another experiment, change the search-space dictionary and/or benchmark settings and rerun.
